<div style="display: flex; align-items: center; padding: 20px; background-color: #f0f2f6; border-radius: 10px; border: 2px solid #007bff;">
    <img src="../logo.png" style="width: 80px; height: auto; margin-right: 20px;">
    <div style="flex: 1; text-align: left;">
    <h1 style="color: #007bff; margin-bottom: 5px;">GLY 6739.017S26: Computational Seismology</h1>
    <h3 style="color: #666;">Notebook 81: Single-channel STA/LTA (Event) Detection</h3>
    <p style="color: red;"><i>Glenn Thompson | Spring 2026</i></p>
    </div>
</div>

# Read the master Stream object we saved to a Pickle file

In [ ]:
from obspy import read
stream_master = read('stream_master.pkl', format='PICKLE')
print(stream_master)


# Introduction to Triggering

Follow the tutorial at:
https://docs.obspy.org/tutorial/code_snippets/trigger_tutorial.html

# Triggering on RSO data from March 20-23, 2009

The examples below are all based on the coincidence network trigger, but we use a module in the week8 folder called 'event_triggering.py', to create a pattern that is probably more useful.

### Find a time window with one obvious event

In [ ]:
t0 = stream_master[0].stats.starttime
st2 = stream_master.copy().trim(
    starttime=t0 + 400,
    endtime=t0 + 550
)

st2.plot();

### Run event triggering on that time window

In [ ]:
from event_triggering import run_trigger_wrapper_df

# We build a pandas DataFrame (a table) of our trigger (detection) windows
df_events = run_trigger_wrapper_df(
    st2,
    sta_seconds=1.0,
    lta_seconds=10.0,
    threshold_on=2.5,
    threshold_off=0.5,
    min_channels=1,
    pretrigger_seconds=10,
    posttrigger_seconds=15,
    write_mseed=False,      # set True to write files
    outdir="events",        # folder for output files
    make_plots=True        # True if you want per-event plots
)

df_events

### Run it on all 4 days!

In [ ]:
import pandas as pd
import event_triggering
from obspy import Stream

list_of_dataframes = []

for tr in stream_master:
    
    df = event_triggering.run_trigger_wrapper_df(
        Stream(traces=[tr]),
        sta_seconds=1.0,
        lta_seconds=10.0,
        threshold_on=2.5,
        threshold_off=0.5,
        min_channels=1,
        pretrigger_seconds=10,
        posttrigger_seconds=15,
        write_mseed=False,      # set True to write files
        outdir="events",        # folder for output files
        make_plots=False        # True if you want per-event plots
    )
    list_of_dataframes.append(df)

df_events = pd.concat(list_of_dataframes, ignore_index=True)
df_events

### Filtering

Since we only used a single channel, a lot of our detections are of dubious quality
We can filter them using:
- min_snr: Minimum signal to noise ratio
- min_coincidence: Not much use here since we only have 1 channel, but if we had a network of 10 Z-channels, we might set this to 5 or 6
- min_duration_s: Very short events could just be interference (noise) spikes

In [ ]:
# df_events: your full candidate catalogue
df_good = event_triggering.filter_events_df(df_events, min_snr=2.0, min_coincidence=1, min_duration_s=5.0)

### Write to MiniSEED files (and make PNG plots)

In [ ]:
# Make a directory to write the Event MiniSEED files to
import os
local_data_dir = os.path.join(os.path.expanduser('~'), 'CompSci_Week8', 'Events')
os.makedirs(local_data_dir, exist_ok=True)

df_exported = event_triggering.export_events_from_catalogue(
    df_good,
    base_outdir=local_data_dir,
    write_mseed=True,
    write_png=True,
    st_continuous=stream_master,  # must cover the event windows
    max_events=50
)

df_exported[["on_time", "snr_rms", "export_mseed_path", "export_png_path"]].head()

# Write the dataframe to a Pickle file
pickle_file = os.path.join(local_data_dir, 'event_index.pkl')
df_exported.to_pickle(pickle_file)

# Write the dataframe to a CSV file
df_exported.to_csv(pickle_file.replace('.pkl', '.csv'))

### Plot hourly event detection rate for ALL events

In [ ]:
fig = event_triggering.plot_event_rate(df_events, bin_size="1H", title="All triggers (hourly)")

### Plot hourly event detection rate for HIGHER QUALITY events

In [ ]:
fig = event_triggering.plot_event_rate(
    df_events,
    bin_size="1H",
    min_snr=2.0,
    min_coincidence=1,
    min_duration_s=8.0,
    title="Filtered triggers (hourly)"
)